In [8]:
#final predictions using the best model
#will integrate the output into the streamlit later

import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [9]:
# Load main dataset
df = pd.read_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/data/final_2020_2026.csv")
df["date"] = pd.to_datetime(df["date"])

# Load model comparison
comparison_df = pd.read_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/outputs/model_comparison.csv")

# Add macro variables again

df["crude_oil_price"] = 0
df["natural_gas_price"] = 0
df["war_index"] = 0

for i in range(len(df)):
    
    date = df.loc[i, "date"]
    
    if date < pd.to_datetime("2026-02-01"):
        df.loc[i, "crude_oil_price"] = 80
        df.loc[i, "natural_gas_price"] = 3
        df.loc[i, "war_index"] = 0
        
    else:
        df.loc[i, "crude_oil_price"] = 110
        df.loc[i, "natural_gas_price"] = 5
        df.loc[i, "war_index"] = 1

In [10]:
results = []

In [11]:
chemicals = df["chemical"].unique()

for chem in chemicals:
    
    temp_df = df[df["chemical"] == chem].copy()
    temp_df = temp_df.sort_values("date")
    temp_df.set_index("date", inplace=True)

    y = temp_df["price_usd_per_ton"].asfreq("MS")
    y = y.dropna()

    # Get best model
    best_model = comparison_df[
        comparison_df["Chemical"] == chem
    ]["Best Model"].values[0]

    print(f"{chem} → {best_model}")

    try:
        if best_model == "ARIMA":
            model = ARIMA(y, order=(1,1,1))
            model_fit = model.fit()
            forecast = model_fit.forecast(steps=1)

        else:
            X = temp_df[["crude_oil_price", "natural_gas_price", "war_index"]].asfreq("MS")
            X = X.loc[y.index]

            model = SARIMAX(
                y,
                exog=X,
                order=(1,1,1),
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            model_fit = model.fit()

            future_X = pd.DataFrame({
                "crude_oil_price": [110],
                "natural_gas_price": [5],
                "war_index": [1]
            }, index=[pd.to_datetime("2026-03-01")])

            forecast = model_fit.forecast(steps=1, exog=future_X)

        results.append({
            "Chemical": chem,
            "Predicted Price (USD/ton)": forecast.values[0]
        })

    except Exception as e:
        print(f"Error in {chem}: {e}")
        results.append({
            "Chemical": chem,
            "Predicted Price (USD/ton)": np.nan
        })

Acetic Acid → SARIMAX
Formates → SARIMAX
KCL → ARIMA
PAC → SARIMAX
Sulphur → SARIMAX
Sulphuric Acid → SARIMAX
Urea → SARIMAX
Water Treatment Chemicals → SARIMAX


/Users/mangalyaphaye/.pyenv/versions/suvira-env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/mangalyaphaye/.pyenv/versions/suvira-env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


In [12]:
final_df = pd.DataFrame(results)
final_df

,Chemical,Predicted Price (USD/ton)
0,Acetic Acid,716.420129
1,Formates,1357.188816
2,KCL,337.351756
3,PAC,381.863269
4,Sulphur,440.113071
5,Sulphuric Acid,258.083467
6,Urea,542.264866
7,Water Treatment Chemicals,4333.889514


In [15]:
final_df["Predicted Price (USD/ton)"] = final_df["Predicted Price (USD/ton)"].round(2)
final_df["Manual Price (USD/ton)"] = None
final_df["Final Price (USD/ton)"] = final_df["Predicted Price (USD/ton)"]

final_df

,Chemical,Predicted Price (USD/ton),Manual Price (USD/ton),Final Price (USD/ton)
0,Acetic Acid,716.42,None,716.42
1,Formates,1357.19,None,1357.19
2,KCL,337.35,None,337.35
3,PAC,381.86,None,381.86
4,Sulphur,440.11,None,440.11
5,Sulphuric Acid,258.08,None,258.08
6,Urea,542.26,None,542.26
7,Water Treatment Chemicals,4333.89,None,4333.89


In [17]:
final_df.to_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/outputs/final_prices_v4_before SentimentAnalysis.csv", index=False)